<a href="https://colab.research.google.com/github/MicaGreen25/TP_IA_Segmentacion_2025/blob/main/TPFinal_IA_Segmentacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# # ============================================================
# #  IMPORTACIONES
# # ============================================================
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

#Pytorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#para segmentacion (U-Net)
!pip install segmentation-models-pytorch
import segmentation_models_pytorch as smp


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.7 MB/s eta 0:00:00


In [ ]:
# # ============================================================
# #  INICIALIZACIONES
# # ============================================================

# numero de clases de Berkeley Deep Drive 100k (clases utiles)
N_CLASSES = 20

# tamaño al que vamos a redimensionar (alto, ancho)
IMG_SIZE = (256, 512)

BATCH_SIZE = 16

# mapeo color RGB -> id de clase
BDD_COLORMAP = {
    (  0,  0,  0): 0,    # fondo       -> negro
    (128, 64,128): 1,   # camino      -> purpura
    (244, 35,232): 2,   # vereda      -> rosa fuerte
    (70, 70, 70):  3,   # edificios   -> gris oscuro
    (102,102,156): 4,   # pared       -> lavanda
    (190,153,153): 5,   # cerca       -> rosa palido
    (153,153,153): 6,   # poste       -> gris clarito
    (250,170, 30): 7,   # semaforo    -> amarillo anaranjado
    (220,220,  0): 8,   # senial de trafico -> amarillo lima
    (107,142, 35): 9,   # vegetacion  -> verde musgo
    (152,251,152): 10,  # terreno     -> verde fluor
    (70,130,180):  11,  # cielo       -> celeste oscuro
    (220, 20, 60): 12,  # persona     -> rojo frambuesa
    (255,  0,  0): 13,  # conductor   -> rojo brillante
    (  0,  0,142): 14,  # auto        -> azul oscuro
    (  0,  0, 70): 15,  # camion      -> azul marino
    (  0, 60,100): 16,  # autobus     -> azul petroleo
    (  0, 80,100): 17,  # tren        -> azul petroleo mas oscuro
    (  0,  0,230): 18,  # moto        -> azul electrico
    (119, 11, 32): 19,  # bicicleta   -> bordo
}

# CLASSES_WEIGHTS =  [0.0043042 , 0.00669084, 0.00484417, 0.02784251, 0.01361132, 0.00443695,
#  0.00964061, 0.00567402, 0.00454273, 0.01086085, 0.00436957, 0.01332698,
#  0.0878623,  0.00429915, 0.01305427, 0.03260668, 0.56434631, 0.11738403,
#  0.06609461, 0.00420792]

# pesos para cross entropy y que se considere el desbalance de clases
CLASSES_WEIGHTS =  [2.19029979e-04, 2.42132302e-03, 3.88809357e-04, 9.99639751e-03,
 4.57454324e-03, 5.53876180e-03, 3.48159108e-02, 1.65355777e-02,
 3.42817624e-04, 4.46025998e-03, 2.48460726e-04, 2.19767639e-02,
 2.12089756e-01, 5.89847355e-04, 4.89716414e-03, 9.68845238e-03,
 4.15070595e-01, 1.73197942e-01, 8.26773233e-02, 2.70263728e-04]



In [ ]:
# # ============================================================
# #  PATHS DEL DATASET
# # ============================================================
!git clone https://github.com/Milagrosgotthelf/muchas_eliminadas.git

# creamos carpetas en Colab para guardar las imagenes
TRAIN_IMAGES ="/content/muchas_eliminadas/train"
TRAIN_MASKS  = "/content/muchas_eliminadas/train_colors"

VAL_IMAGES   = "/content/muchas_eliminadas/val"
VAL_MASKS    = "/content/muchas_eliminadas/val_colors"

TEST_IMAGES  = "/content/muchas_eliminadas/test"

Cloning into 'muchas_eliminadas'...
remote: Enumerating objects: 9009, done.
remote: Total 9009 (delta 0), reused 0 (delta 0), pack-reused 9009 (from 2)
Receiving objects: 100% (9009/9009), 633.29 MiB | 34.87 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Updating files: 100% (9001/9001), done.


In [ ]:
from PIL import Image
# se tiene un BarkeleyDeepDriveGenerator para tarin, val y test
class BerkeleyDeepDriveGenerator(Dataset):
  # guardamos las rutas y listas de ficheros
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

        self.image_files = sorted(os.listdir(images_dir)) #se ordena alfabeticamente la lista de imagenes del directorio

        # Si NO hay mascaras es caso test
        if masks_dir is None:
            self.mask_files = None
        else:
            self.mask_files = sorted(os.listdir(masks_dir)) #se ordena alfabeticamente la lista de mascaras del directorio
            assert len(self.image_files) == len(self.mask_files), "Desbalance entre cantidad de imagenes y mascaras"

    def __len__(self):
        return len(self.image_files)

    def preprocess_mask(self, mask):
        mask_np = np.array(mask)  # [H,W,3]
        h, w, _ = mask_np.shape
        result = np.zeros((h, w), dtype=np.uint8) #matriz de hxw

        for color, class_id in BDD_COLORMAP.items():
            matches = np.all(mask_np == color, axis = -1)
            result[matches] = class_id

        return result

    def __getitem__(self, idx):
      #SACAR!!!!!!!!!!!!! ES SOLO PARA VER QUE FUNCIONE
        if idx % 200 == 0:
          print("Leyendo sample:", idx)
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        # si es de train o val
        if self.masks_dir is not None:
            base = os.path.splitext(img_name)[0]
            #arammos el nombre correcto de la masacra
            mask_name = base + "_train_color.png"

            mask_path = os.path.join(self.masks_dir, mask_name)
            mask = Image.open(mask_path).convert("RGB")

            # color -> clase
            mask = self.preprocess_mask(mask)


        img_np = np.array(img)
        if self.transform:
            if mask is not None:
                # esto aegura que si se rota, recorta, resize o flip la imagen, tambien lo hara su mascara
                transformed = self.transform(image=img_np, mask=mask)
                img = transformed["image"]
                mask = transformed["mask"]
            else:
                img = self.transform(image=img_np)["image"]

        if self.masks_dir is None:
          return img
        else:
          return img, mask


In [ ]:

# ============================================================
# Transforms (usamos Albumentations para augmentations)
# ============================================================

import albumentations as A
from albumentations.pytorch import ToTensorV2 # convierte imágenes/máscaras en tensores PyTorch (shape C×H×W)

# A.Compose recibe una lisat de transformaciones que se aplican secuencialmente
train_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]), # entradas conscistentes y que pueda procesar
    A.HorizontalFlip(p=0.5), # invierte la imagen horizontalmente con prob 50% -> duplica el dato -> auto mirando a derecha y auto mirando a izquierda
    A.RandomBrightnessContrast(p=0.3), # modifica el brillo y contraste simulando dias nublados, sombras, cambios de luz en la calle
    A.GaussianBlur(p=0.2), # es un leve desenfoque que simula imagens en movimiento, un foco danado o camara con vibracion
    A.Normalize(), # convierte los valores de la imagen a media 0 y varianza 1
    ToTensorV2() # de H×W×C (Numpy) a C×H×W (PyTorch) y de unit8 a float32
], is_check_shapes=False)

# estas imagnes si deben representar la realidad del dataset por eso no las alteramos con aumentaciones
val_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(),
    ToTensorV2()
],is_check_shapes=False)

test_transform = A.Compose([
    A.Resize(IMG_SIZE[0], IMG_SIZE[1]),
    A.Normalize(),
    ToTensorV2()
], is_check_shapes=False)

In [ ]:
# # ============================================================
# # DataLoaders
# # ============================================================

train_dataset = BerkeleyDeepDriveGenerator(TRAIN_IMAGES, TRAIN_MASKS, transform=train_transform)
val_dataset   = BerkeleyDeepDriveGenerator(VAL_IMAGES, VAL_MASKS, transform=val_transform)
test_dataset  = BerkeleyDeepDriveGenerator(TEST_IMAGES, None, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset, batch_size=1)


In [ ]:
# # ============================================================
# # U-Net (segmentation_models_pytorch)
# # ============================================================
model = smp.Unet( encoder_name="resnet34",
                  encoder_weights="imagenet",
                  classes=N_CLASSES,
                  activation=None
                )
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
# criterion = nn.CrossEntropyLoss()
classes_weights_tensor = torch.tensor(CLASSES_WEIGHTS, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=classes_weights_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
import numpy as np
from tqdm import tqdm
pixel_counts = np.zeros(N_CLASSES, dtype=np.int64)
for idx in tqdm(range(len(train_dataset))):
    _, mask = train_dataset[idx]   # la máscara ya viene con clases 0..19
    mask_np = mask.numpy()         # [H, W]
    for c in range(N_CLASSES):
        pixel_counts[c] += np.sum(mask_np == c)

# Inversamente proporcional a la frecuencia
class_weights = 1.0 / (pixel_counts + 1e-6)
class_weights = class_weights / class_weights.sum()  # normalizar
print("CLASSES_FREQ = ", pixel_counts)
print("CLASSES_WEIGHTS = ", class_weights)


  0%|          | 0/3500 [00:00<?, ?it/s]

Leyendo sample: 0


  6%|▌         | 200/3500 [01:51<28:36,  1.92it/s]

Leyendo sample: 200


 11%|█▏        | 400/3500 [03:41<26:47,  1.93it/s]

...

Leyendo sample: 3200


 97%|█████████▋| 3400/3500 [31:03<01:08,  1.46it/s]

Leyendo sample: 3400


100%|██████████| 3500/3500 [31:57<00:00,  1.83it/s]

CLASSES_FREQ =  [80529588 99366428  8988568 55976602  2177207  4757683  3929439   625123
  1316206 63486312  4879587 87596245   990329   102618 36898066  4444251
  2246409    52435   125661   263243]
CLASSES_WEIGHTS =  [2.70263728e-04 2.19029979e-04 2.42132302e-03 3.88809357e-04
 9.99639751e-03 4.57454324e-03 5.53876180e-03 3.48159108e-02
 1.65355777e-02 3.42817624e-04 4.46025998e-03 2.48460726e-04
 2.19767639e-02 2.12089756e-01 5.89847355e-04 4.89716414e-03
 9.68845238e-03 4.15070595e-01 1.73197942e-01 8.26773233e-02]


In [ ]:
# ============================================================
# Entrenamiento
# ============================================================

most_significant_classes = {
    11: "person",
    12: "rider",
    13: "car",
    14: "truck",
    15: "bus",
    17: "motorcycle",
    18: "bicycle"
}

def train_epoch(loader):
    model.train()
    running_loss = 0
    #DEJAR EL FOR SOLO CON imgs, masks
    for batch_idx, (imgs, masks) in enumerate(loader):
      #SACAR!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        if batch_idx % 50 == 0:
            print(f"Leyendo batch {batch_idx}/{len(loader)}")
        imgs = imgs.to(device)
        masks = masks.to(device).long()

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)


def val_epoch(loader):
    model.eval()
    running_loss = 0
    all_ious = []

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device).long()

            outputs = model(imgs)
            loss = criterion(outputs, masks)
            running_loss += loss.item()

            pred_masks = torch.argmax(outputs, dim=1);

            for p, t in zip(pred_masks, masks):
                ious, miou = compute_mIoU(p.cpu(), t.cpu(), n_classes=N_CLASSES)
                all_ious.append(ious)

    # mean_iou = np.nanmean(all_ious)
    mean_ious_per_class = np.nanmean(all_ious, axis=0)
    mean_miou = np.nanmean(mean_ious_per_class)

    # Solo las clases más significativas
    ious_significant = [mean_ious_per_class[idx] for idx in most_significant_classes.keys()]

    return running_loss / len(loader), mean_miou, mean_ious_per_class, ious_significant


def compute_mIoU(pred_mask, true_mask, n_classes=N_CLASSES):
    """
    pred_mask: [H, W] predicciones de clase por pixel
    true_mask: [H, W] máscara real
    Devuelve IoU por clase y mIoU
    """
    ious = []
    pred_mask = pred_mask.view(-1)
    true_mask = true_mask.view(-1)

    for cls in range(n_classes):
        pred_inds = pred_mask == cls
        target_inds = true_mask == cls

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            iou = float('nan')  # ignorar clase ausente
        else:
            iou = intersection / union
        ious.append(iou)

    miou = np.nanmean(ious)  # promedio ignorando NaN
    return ious, miou

# from google.colab import drive
# drive.mount('/content/drive')

# import glob
# import os
# import torch

# checkpoint_path = "/content/drive/MyDrive/checkpoints_unet"
# os.makedirs(checkpoint_path, exist_ok=True)

# best_model_path = checkpoint_path + "/best_model_mIoU.pth"

# # REAUNUDAR DESDE EL ULTIMO CHECKPOINT POR FALTA DE RECURSOS INFORMATICOS
# ckpts = sorted(glob.glob(f"{checkpoint_path}/epoch_*.pth"))

# if ckpts:
#     last_ckpt = ckpts[-1]
#     print("Cargando último checkpoint:", last_ckpt)
#     model.load_state_dict(torch.load(last_ckpt, map_location=device))
#     start_epoch = int(last_ckpt.split("_")[-1].split(".")[0])
#     print(f"Entrenamiento continuará desde la epoch {start_epoch}")

#     # Restaurar mejor modelo si existe
#     if os.path.exists(best_model_path):
#         print("Restaurando mejor mIoU previo…")
#         best_state = torch.load(best_model_path, map_location=device)
#         best_miou = best_state["best_miou"]  # ← guardado como metadata
#     else:
#         best_miou = -1

# else:
#     print("No hay checkpoints previos. Entrenamiento desde cero.")
#     start_epoch = 0
#     best_miou = -1

# EPOCHS = 10

# train_losses, val_losses = [], []
# val_losses, val_mious, val_ious_all, val_ious_significant_list = [], [], [], []


# for epoch in range(start_epoch, EPOCHS):
#     train_loss = train_epoch(train_loader)
#     val_loss, val_miou, ious_all, ious_significant = val_epoch(val_loader)

#     train_losses.append(train_loss)
#     val_losses.append(val_loss)
#     val_mious.append(val_miou)
#     val_ious_all.append(ious_all)               # IoU de todas las clases
#     val_ious_significant_list.append(ious_significant)  # Solo las clases importantes


#     print(f"Epoch {epoch+1}/{EPOCHS}")
#     print(f"  Train Loss: {train_loss:.4f}")
#     print(f"  Val   Loss: {val_loss:.4f}")
#     print(f"  Val mIoU  : {val_miou:.4f}")

#     # guardamos mejora en modelo
#     if val_miou > best_miou:
#         print(f">>>>>> Nuevo mejor modelo guardado: val_miou mejoro {(val_miou - best_miou):.6f}")
#         best_miou = val_miou
#         torch.save(
#             {
#                 "model_state": model.state_dict(),
#                 "best_miou": best_miou
#             },
#             best_model_path
#         )

#     # guardamos el checkpoint
#     save_file = f"{checkpoint_path}/epoch_{epoch+1}.pth"
#     torch.save(model.state_dict(), save_file)
#     print(f"Checkpoint guardado en: {save_file}")


In [ ]:
# ============================================================
# Graficar métricas
# ============================================================
# plt.figure(figsize=(8,5))
# plt.plot(train_losses, label="Train Loss")
# plt.plot(val_losses, label="Val Loss")
# plt.xlabel("Epoch")
# plt.ylabel("Loss")
# plt.title("Loss por época")
# plt.legend()
# plt.grid(True)
# plt.show()

# plt.figure(figsize=(10,5))
# plt.subplot(1,2,1)
# plt.plot(val_mious, label="Val mIoU", color="green")
# plt.xlabel("Epoch")
# plt.ylabel("mIoU")
# plt.title("mIoU global por época")
# plt.legend()

# plt.subplot(1,2,2)
# ious_array = np.array(val_ious_significant_list)
# for i, cls_name in enumerate(most_significant_classes.values()):
#     plt.plot(range(1, EPOCHS+1), ious_array[:, i], label=cls_name)
# plt.xlabel("Epoch")
# plt.ylabel("IoU")
# plt.title("IoU clases significativas")
# plt.legend()

# plt.tight_layout()
# plt.show()

In [ ]:
#=========VER SI ESTA FUNCIONA O PONGO LA DEL OTRO PROGRAMA=======
def denormalize(img_tensor):
    """
    img_tensor: tensor [3,H,W] en rango [0,1] porque usaste A.Normalize() sin mean/std.
    """
    img = img_tensor.permute(1,2,0).cpu().numpy()  # [H,W,3]
    img = (img * 255).clip(0,255).astype("uint8")   # volver a [0-255]
    return img


In [ ]:
# ============================================================
# Predicción sobre TEST
# ============================================================
import shutil

output_dir = "test_pred_masks"   # se creará dentro de /content/
drive_output_dir = "/content/drive/MyDrive/BDD100K_predicciones/"
os.makedirs(drive_output_dir, exist_ok=True)


# Si existe, borrarla completa
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

# Crear carpeta vacía
os.makedirs(output_dir, exist_ok=True)

# DataLoader
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model.eval()
model.to(device)

with torch.no_grad():
    for idx, img in enumerate(test_loader):

        # Imagen al device
        img = img.to(device)   # tensor [1,3,H,W] (batch, canales, alto, ancho)

        # Obtener el nombre REAL del archivo (sin modificar __getitem__)
        real_name = test_dataset.image_files[idx]
        base = os.path.splitext(real_name)[0]           # sin el .png
        mask_name = base + "_mask.png"                  # para poder identificar las mascaras con las imagenes originales de tets, le ponemos el mismo nombre con _mask

        # ---- Predicción ----
        pred = model(img)                                      # [1, C, H, W]
        pred_mask = torch.argmax(pred, dim=1).cpu().numpy()[0] # [H,W]
        # con argmax se elige la clase con mayor probabilidad por pixel y pasamos a [1,H,W]
        # .cpu() lo pasa del gpu al cpu
        # .numpy() lo ocnvierte en un array NumPy
        # [0] le saca la dimension del batch

        # ---- Guardar máscara ----
        mask_img = Image.fromarray(pred_mask.astype(np.uint8))
        mask_img.save(os.path.join(output_dir, mask_name))
        mask_img.save(os.path.join(drive_output_dir, mask_name))

        # ---- Mostrar solo primeras 10 ----
        if idx < 10:

            #img_np = img.squeeze().permute(1,2,0).cpu().numpy()
            # .squeeze() elimina la dimension del batch
            # .permute(1,2,0) pasa a formato imagen normal [H,W,3]
            # .numpy() convierte a NumPy

            #img_np = (img_np * 255).clip(0,255).astype("uint8")
            # se recupera la imagen original antes de la normalizacion por eso se
            # multiplica por 255 para desnormalizar
            # .clip(0,255) para evitar valores fuera de rango
            # unit8 formato estandar de imagen

            img_np = denormalize(img.squeeze())

            plt.figure(figsize=(10,4))
            plt.suptitle(f"Muestra test #{idx} - {real_name}")

            plt.subplot(1,2,1)
            plt.title("Imagen original")
            plt.imshow(img_np)
            plt.axis("off")

            plt.subplot(1,2,2)
            plt.title("Máscara predicha")
            BDD_PALETTE = np.array(list(BDD_COLORMAP.keys()))
            color_mask = BDD_PALETTE[pred_mask]   # [H,W,3]
            plt.imshow(color_mask)
            plt.axis("off")

            plt.show()


In [ ]:
# # ============================================================
# # Ejemplo de predicción
# # ============================================================
# import shutil
# output_dir = "test_pred_masks"

# # Si existe, borrar todo lo que contiene
# if os.path.exists(output_dir):
#     shutil.rmtree(output_dir)

# # Crear la carpeta nuevamente (vacía)
# os.makedirs(output_dir)
# test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# model.eval()
# model.to(device)

# with torch.no_grad(): #no hay gardientes porque es test, no train
#     for idx, sample in enumerate(test_loader): #sample es un diccionario con image, name

#         img = sample["image"].to(device)    # tensor [1,3,H,W] (batch, canales, alto, ancho)
#         img_name =  f"test_{idx}.png"      # si llegamos arreglar para que coincida realmente con el nombre real

#         # ---- Predicción ----
#         pred = model(img)                   # [1, C, H, W]
#         pred_mask = torch.argmax(pred, dim=1).cpu().numpy()[0]  # [H,W]
#         # con argmax se elige la clase con mayor probabilidad por pixel y pasamos a [1,H,W]
#         # .cpu() lo pasa del gpu al cpu
#         # .numpy() lo ocnvierte en un array NumPy
#         # [0] le saca la dimension del batch

#         # ---- Guardar máscara como imagen ----
#         mask_img = Image.fromarray(pred_mask.astype(np.uint8))
#         mask_img.save(os.path.join(save_dir, img_name.replace(".jpg", "_mask.png")))
#         # para poder identificar las mascaras con las imagenes originales de tets, le ponemos el mismo nombre con _mask

#         # ---- Mostrar solo las primeras 10 ----
#         if idx < 10:

#             # Convertir imagen normalizada a formato visible
#             img_np = sample["image"].squeeze().permute(1,2,0).numpy()
#             # .squeeze() elimina la dimension del batch
#             # .permute(1,2,0) pasa a formato imagen normal [H,W,3]
#             # .numpy() convierte a NumPy

#             img_np = (img_np * 255).clip(0,255).astype("uint8")
#             # se recupera la imagen original antes de la normalizacion por eso se
#             # multiplica por 255 para desnormalizar
#             # .clip(0,255) para evitar valores fuera de rango
#             # unit8 formato estandar de imagen

#             # Plot lado a lado
#             plt.figure(figsize=(10,4))
#             plt.suptitle(f"Muestra test #{idx}")

#             plt.subplot(1,2,1)
#             plt.title("Imagen original")
#             plt.imshow(img_np)
#             plt.axis("off")

#             plt.subplot(1,2,2)
#             plt.title("Mascara predicha")
#             plt.imshow(pred_mask, cmap="tab20")
#             plt.axis("off")

#             plt.show()
